# Задание 4: аугментация и оптимизация одиночной сети

Это самостоятельный эксперимент на архитектуре, выбранной в `task4_regularization.ipynb`:
90 →128 →128 →1, BatchNorm, ReLU, dropout0.55, AdamW weight decay0.5.
Исходный `task4.ipynb` не изменён. Используются только выданные CSV, все модели/лоссы/оптимизация — torch.
Никаких ансамблей, pretrained или сторонних данных. Audit не оцениваем.
Проверяем mixup, Gaussian noise, маскирование признаков, Adam betas, learning rate и размер батча.
Все аугментации действуют только на training batch; validation и test проходят без них.

**Обновление после завершения поиска:** в последнем приложении добавлена однократная audit-оценка зафиксированных кандидатов; во время поиска audit не использовался.


In [2]:
import base64
import copy
import hashlib
import io
import math
import random
import time
import unittest
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import json
import pickle
import torch
from IPython.display import HTML, Markdown, display
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, RobustScaler, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
SELECTION_METRIC = "rmse"  # "rmse", "mse" или "mae"; см. пояснение выше.
assert SELECTION_METRIC in {"rmse", "mse", "mae"}
MAX_EPOCHS = 160
PATIENCE = 25
BATCH_SIZE = 256
# CPU выбран для воспроизводимого сравнения двух реализаций оптимизатора.
# Для большого запуска можно указать "cuda" или "mps", если устройство доступно.
DEVICE = torch.device("cpu")
torch.set_num_threads(2)


def seed_everything(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)


seed_everything()
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})
print(f"torch={torch.__version__}; numpy={np.__version__}; "
      f"pandas={pd.__version__}; sklearn={sklearn.__version__}; device={DEVICE}")

torch=2.14.1; numpy=2.4.6; pandas=3.0.3; sklearn=1.9.0; device=cpu


In [3]:
candidates = [Path("task4_data"), Path("homeworks/task4_data")]
DATA_DIR = next((p for p in candidates if (p / "train_x.csv").is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Запустите ноутбук из корня репозитория или из homeworks.")

train_x = pd.read_csv(DATA_DIR / "train_x.csv", index_col=0)
train_y = pd.read_csv(DATA_DIR / "train_y.csv", index_col=0)
test_table = pd.read_csv(DATA_DIR / "test_x.csv")
assert train_x.index.is_unique and train_y.index.is_unique
assert set(train_x.index) == set(train_y.index), "Индексы признаков и меток не совпадают"
train_y = train_y.loc[train_x.index]
assert list(train_y.columns) == ["year"]
assert "id" in test_table and test_table["id"].is_unique
assert not test_table["id"].isna().any()

feature_columns = list(train_x.columns)
assert len(feature_columns) == 90
assert set(test_table.columns) == set(feature_columns) | {"id"}
test_ids = test_table["id"].copy()
test_x = test_table.loc[:, feature_columns].copy()
assert all(pd.api.types.is_numeric_dtype(t) for t in train_x.dtypes)
assert all(pd.api.types.is_numeric_dtype(t) for t in test_x.dtypes)
X = train_x.to_numpy(dtype=np.float64)
X_test = test_x.to_numpy(dtype=np.float64)
y = torch.tensor(train_y["year"].to_numpy(dtype=np.float32))
assert torch.isfinite(y).all() and len(y) == len(X)
assert torch.equal(y, y.round()), "Годы должны быть целыми"
assert not np.isinf(X).any() and not np.isinf(X_test).any()

print(f"Обучение: {X.shape}; тест: {X_test.shape}; целевая переменная: year")
print(f"Пропуски: train={np.isnan(X).sum()}, test={np.isnan(X_test).sum()}")
print(f"Точные дубликаты train: {train_x.duplicated().sum()}")
for name in ("train_x.csv", "train_y.csv", "test_x.csv"):
    print(name, "SHA256:", hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest())

Обучение: (14000, 90); тест: (6000, 90); целевая переменная: year
Пропуски: train=0, test=0
Точные дубликаты train: 0
train_x.csv SHA256: aab2f64a4fdb0fd6af98ed95fedc9a3d9a54506b1f607da725926f654f5bff89
train_y.csv SHA256: 1f72519091e6e08948c430a1db9b181df28095e0cc9f29694570b40daed683bf
test_x.csv SHA256: 54f91d13c6f61fd634bae76bdad109471585a4ce0b308159be16f768134b4046


In [4]:
# NumPy/sklearn здесь применяются только к предварительному разбиению данных.
periods = np.digitize(y.tolist(), bins=[1970, 1980, 1990, 2000, 2005])
indices = np.arange(len(X))
development_idx, audit_idx = train_test_split(
    indices, test_size=0.15, random_state=SEED, stratify=periods
)
train_idx, valid_idx = train_test_split(
    development_idx, test_size=0.15 / 0.85, random_state=SEED,
    stratify=periods[development_idx]
)
assert not (set(train_idx) & set(valid_idx) or set(train_idx) & set(audit_idx)
            or set(valid_idx) & set(audit_idx))
assert len(set(train_idx) | set(valid_idx) | set(audit_idx)) == len(X)
row_hashes = pd.util.hash_pandas_object(train_x, index=False)
assert not (set(row_hashes.iloc[train_idx]) & set(row_hashes.iloc[valid_idx]))
assert not (set(row_hashes.iloc[development_idx]) & set(row_hashes.iloc[audit_idx]))
y_train, y_valid = y[train_idx], y[valid_idx]
print(f"train={len(train_idx)}, validation={len(valid_idx)}, audit={len(audit_idx)}")
print(f"Train: mean={y_train.mean():.2f}, std={y_train.std():.2f}, "
      f"min={y_train.min():.0f}, max={y_train.max():.0f}")


train=9800, validation=2100, audit=2100
Train: mean=1998.37, std=11.07, min=1922, max=2011


In [5]:
class FeaturePreprocessor:
    def __init__(self, kind="standard"):
        if kind not in {"standard", "asinh", "quantile"}:
            raise ValueError(kind)
        self.kind = kind
        self.imputer = SimpleImputer(strategy="median", keep_empty_features=True)

    def fit(self, x):
        z = self.imputer.fit_transform(x)
        if self.kind == "asinh":
            self.robust = RobustScaler().fit(z)
            z = np.arcsinh(self.robust.transform(z))
        self.scaler = (
            QuantileTransformer(n_quantiles=min(256, len(z)), output_distribution="normal",
                                subsample=None, random_state=SEED)
            if self.kind == "quantile" else StandardScaler()
        )
        self.scaler.fit(z)
        return self

    def transform(self, x):
        z = self.imputer.transform(x)
        if self.kind == "asinh":
            z = np.arcsinh(self.robust.transform(z))
        z = self.scaler.transform(z).astype(np.float32)
        if not np.isfinite(z).all():
            raise ValueError("Нефинитные признаки после преобразования")
        return torch.from_numpy(z.copy())


class TargetScaler:
    def __init__(self, target):
        self.mean = float(target.mean())
        self.std = max(float(target.std(unbiased=False)), 1e-6)

    def encode(self, target):
        return (target - self.mean) / self.std

    def decode(self, prediction):
        return prediction * self.std + self.mean


@torch.inference_mode()
def regression_metrics(prediction, target):
    assert prediction.shape == target.shape
    error = prediction.double() - target.double()
    mse = error.square().mean().item()
    return {"rmse": math.sqrt(mse), "mse": mse, "mae": error.abs().mean().item()}


def print_metrics(label, values):
    print(f"{label}: RMSE={values['rmse']:.4f}; MSE={values['mse']:.4f}; MAE={values['mae']:.4f}")


constant = y_train.median() if SELECTION_METRIC == "mae" else y_train.mean()
constant_valid = regression_metrics(torch.full_like(y_valid, float(constant)), y_valid)
print_metrics("Константа, validation", constant_valid)

Константа, validation: RMSE=10.9058; MSE=118.9360; MAE=8.1448


## Оптимизатор и точный baseline

Сохраняем собственную реализацию AdamW, её тесты и исходную residual-архитектуру.
Для baseline используются его DataLoader, расписание learning rate (160 эпох) и patience=25.
Источники формулы AdamW: [статья](https://arxiv.org/abs/1711.05101),
[документация PyTorch](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html).
Основы цикла обучения: локальный `notebooks/pytorch_intro.ipynb`.


In [7]:
class CustomAdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)
        for group in self.param_groups:
            if not math.isfinite(group["lr"]) or group["lr"] < 0:
                raise ValueError("lr должен быть конечным и неотрицательным")
            if not math.isfinite(group["eps"]) or group["eps"] <= 0:
                raise ValueError("eps должен быть конечным и положительным")
            if not math.isfinite(group["weight_decay"]) or group["weight_decay"] < 0:
                raise ValueError("weight_decay должен быть конечным и неотрицательным")
            if len(group["betas"]) != 2 or not all(0 <= b < 1 for b in group["betas"]):
                raise ValueError("betas должны лежать в [0, 1)")

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            beta1, beta2 = group["betas"]
            for parameter in group["params"]:
                grad = parameter.grad
                if grad is None:
                    continue
                if grad.is_sparse or torch.is_complex(parameter):
                    raise RuntimeError("Поддерживаются только плотные вещественные градиенты")
                state = self.state[parameter]
                if not state:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(parameter)
                    state["exp_avg_sq"] = torch.zeros_like(parameter)
                state["step"] += 1
                m, v = state["exp_avg"], state["exp_avg_sq"]
                m.lerp_(grad, 1 - beta1)
                v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                correction1 = 1 - beta1 ** state["step"]
                correction2 = 1 - beta2 ** state["step"]
                denominator = v.sqrt().div_(math.sqrt(correction2)).add_(group["eps"])
                parameter.mul_(1 - group["lr"] * group["weight_decay"])
                parameter.addcdiv_(m, denominator, value=-group["lr"] / correction1)
        return loss

In [8]:
class TestCustomAdamW(unittest.TestCase):
    def test_reference_steps_and_parameter_groups(self):
        generator = torch.Generator().manual_seed(101)
        ours = [nn.Parameter(torch.randn(3, 4, generator=generator, dtype=torch.float64)),
                nn.Parameter(torch.randn(4, generator=generator, dtype=torch.float64))]
        reference = [nn.Parameter(p.detach().clone()) for p in ours]
        def groups(parameters):
            return [{"params": [parameters[0]], "lr": 0.003, "weight_decay": 0.1},
                    {"params": [parameters[1]], "lr": 0.002, "weight_decay": 0.0}]
        custom = CustomAdamW(groups(ours), betas=(0.8, 0.95), eps=1e-6)
        builtin = torch.optim.AdamW(groups(reference), betas=(0.8, 0.95), eps=1e-6,
                                    foreach=False, fused=False)
        for step in range(20):
            for i, (p, q) in enumerate(zip(ours, reference)):
                grad = torch.randn(p.shape, generator=generator, dtype=p.dtype)
                if step % 4 == 0 and i == 1:
                    p.grad = q.grad = None
                else:
                    p.grad = torch.zeros_like(grad) if step == 7 else grad
                    q.grad = p.grad.clone()
            custom.step()
            builtin.step()
            for p, q in zip(ours, reference):
                torch.testing.assert_close(p, q, rtol=1e-11, atol=1e-12)

    def test_none_vs_zero_gradient(self):
        p = nn.Parameter(torch.tensor([2.0], dtype=torch.float64))
        optimizer = CustomAdamW([p], lr=0.1, weight_decay=0.2)
        optimizer.step()
        self.assertEqual(p.item(), 2.0)
        self.assertNotIn(p, optimizer.state)
        p.grad = torch.zeros_like(p)
        optimizer.step()
        self.assertAlmostEqual(p.item(), 1.96)
        self.assertEqual(optimizer.state[p]["step"], 1)

    def test_restore_state(self):
        p = nn.Parameter(torch.tensor([1.0, -2.0], dtype=torch.float64))
        a = CustomAdamW([p])
        p.grad = torch.tensor([0.3, -0.1], dtype=p.dtype)
        a.step()
        q = nn.Parameter(p.detach().clone())
        b = CustomAdamW([q])
        b.load_state_dict(copy.deepcopy(a.state_dict()))
        for _ in range(5):
            p.grad = torch.tensor([-0.2, 0.4], dtype=p.dtype)
            q.grad = p.grad.clone()
            a.step()
            b.step()
        torch.testing.assert_close(p, q, rtol=0, atol=0)

    def test_invalid_settings(self):
        for kwargs in [{"lr": -1}, {"eps": 0}, {"betas": (1, 0.9)}, {"weight_decay": -1}]:
            with self.assertRaises(ValueError):
                CustomAdamW([nn.Parameter(torch.ones(1))], **kwargs)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCustomAdamW)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()

test_invalid_settings (__main__.TestCustomAdamW.test_invalid_settings) ... ok
test_none_vs_zero_gradient (__main__.TestCustomAdamW.test_none_vs_zero_gradient) ... ok
test_reference_steps_and_parameter_groups (__main__.TestCustomAdamW.test_reference_steps_and_parameter_groups) ... ok
test_restore_state (__main__.TestCustomAdamW.test_restore_state) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.016s

OK


In [9]:
class ResidualBlock(nn.Module):
    def __init__(self, width, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(width), nn.Linear(width, width * 2), nn.SiLU(),
            nn.Dropout(dropout), nn.Linear(width * 2, width), nn.Dropout(dropout)
        )
        nn.init.zeros_(self.net[-2].weight)
        nn.init.zeros_(self.net[-2].bias)

    def forward(self, x):
        return x + self.net(x)


class YearRegressor(nn.Module):
    def __init__(self, n_features, architecture="residual", width=128, dropout=0.15):
        super().__init__()
        if architecture == "linear":
            self.net = nn.Linear(n_features, 1)
        elif architecture == "mlp":
            self.net = nn.Sequential(
                nn.Linear(n_features, width), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(width, width), nn.ReLU(), nn.Dropout(dropout), nn.Linear(width, 1)
            )
        elif architecture == "residual":
            self.net = nn.Sequential(
                nn.Linear(n_features, width), nn.SiLU(),
                ResidualBlock(width, dropout), ResidualBlock(width, dropout),
                nn.LayerNorm(width), nn.SiLU(), nn.Linear(width, 1)
            )
        else:
            raise ValueError(architecture)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def build_optimizer(model, kind, lr, weight_decay):
    decay, no_decay = [], []
    for parameter in model.parameters():
        (decay if parameter.ndim >= 2 else no_decay).append(parameter)
    groups = [{"params": decay, "weight_decay": weight_decay},
              {"params": no_decay, "weight_decay": 0.0}]
    if kind == "custom":
        return CustomAdamW(groups, lr=lr)
    if kind == "builtin":
        return torch.optim.AdamW(groups, lr=lr, foreach=False, fused=False)
    raise ValueError(kind)


@torch.inference_mode()
def predict(model, features, target_scaler, batch_size=1024):
    model.eval()
    outputs = [model(batch.to(DEVICE)).cpu() for batch in features.split(batch_size)]
    return target_scaler.decode(torch.cat(outputs))


def make_loader(features, targets, seed):
    return DataLoader(TensorDataset(features, targets), batch_size=BATCH_SIZE, shuffle=True,
                      generator=torch.Generator().manual_seed(seed), num_workers=0)


def train_epoch(model, loader, optimizer):
    model.train()
    loss_fn = nn.L1Loss() if SELECTION_METRIC == "mae" else nn.MSELoss()
    total_loss, total_grad, count = 0.0, 0.0, 0
    for features, targets in loader:
        features, targets = features.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        prediction = model(features)
        assert prediction.shape == targets.shape
        loss = loss_fn(prediction, targets)
        if not torch.isfinite(loss):
            raise RuntimeError("Обучение разошлось")
        loss.backward()
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0,
                                           error_if_nonfinite=True)
        optimizer.step()
        total_loss += loss.item() * len(features)
        total_grad += float(grad_norm) * len(features)
        count += len(features)
    return total_loss / count, total_grad / count


def learning_rate(config, epoch):
    # Та же функция используется при final refit, без изменения горизонта.
    phase = (epoch - 1) / max(1, MAX_EPOCHS - 1)
    return config["lr"] * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * phase)))


def fit_experiment(config, optimizer_kind="builtin", seed=SEED):
    seed_everything(seed)
    preprocess = FeaturePreprocessor(config["preprocessing"]).fit(X[train_idx])
    xt, xv = preprocess.transform(X[train_idx]), preprocess.transform(X[valid_idx])
    target_scaler = TargetScaler(y_train)
    loader = make_loader(xt, target_scaler.encode(y_train), seed)
    model = YearRegressor(xt.shape[1], config["architecture"], config["width"],
                          config["dropout"]).to(DEVICE)
    optimizer = build_optimizer(model, optimizer_kind, config["lr"], config["weight_decay"])
    best_score, best_epoch, best_state, stale = math.inf, 0, None, 0
    history = []
    started = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        lr = learning_rate(config, epoch)
        for group in optimizer.param_groups:
            group["lr"] = lr
        loss, grad_norm = train_epoch(model, loader, optimizer)
        train_metrics = regression_metrics(predict(model, xt, target_scaler), y_train)
        valid_metrics = regression_metrics(predict(model, xv, target_scaler), y_valid)
        score = valid_metrics[SELECTION_METRIC]
        history.append({"epoch": epoch, "loss": loss, "grad_norm": grad_norm, "lr": lr,
                        **{f"train_{k}": v for k, v in train_metrics.items()},
                        **{f"valid_{k}": v for k, v in valid_metrics.items()}})
        if score < best_score:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
        if epoch == 1 or epoch % 20 == 0:
            print(f"{config['name']} / {optimizer_kind}: epoch={epoch:3d}, "
                  f"train RMSE={train_metrics['rmse']:.4f}, valid RMSE={valid_metrics['rmse']:.4f}",
                  flush=True)
        if stale >= PATIENCE:
            break
    assert best_state is not None
    model.load_state_dict(best_state)
    predictions = predict(model, xv, target_scaler)
    result = dict(config=dict(config), optimizer=optimizer_kind, model=model.cpu(),
                  preprocessing=preprocess, target_scaler=target_scaler, history=history,
                  best_epoch=best_epoch, metrics=regression_metrics(predictions, y_valid),
                  valid_prediction=predictions, seconds=time.perf_counter() - started)
    print_metrics(f"Лучший checkpoint ({best_epoch} эпоха)", result["metrics"])
    return result

## Общая модель и обучение

Сохраняем один и тот же backbone. Препроцессор обучается только по train данного разбиения.
Все варианты имеют cosine horizon160, максимум 160 эпох, patience20, gradient clipping5.


In [11]:
class DenseRegressor(nn.Module):
    def __init__(self, n_features, config):
        super().__init__()
        width, depth = config["width"], config["depth"]
        norm, dropout = config["norm"], config["dropout"]
        activation = {"relu": nn.ReLU, "silu": nn.SiLU, "tanh": nn.Tanh}[config["activation"]]
        layers = []
        if config.get("input_dropout", 0):
            layers.append(nn.Dropout(config["input_dropout"]))
        previous = n_features
        for _ in range(depth):
            layers.append(nn.Linear(previous, width))
            if norm == "batch": layers.append(nn.BatchNorm1d(width))
            if norm == "layer": layers.append(nn.LayerNorm(width))
            layers += [activation(), nn.Dropout(dropout)]
            previous = width
        layers.append(nn.Linear(previous, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)


def make_model(config, n_features=90):
    if config["architecture"] == "residual":
        return YearRegressor(n_features, "residual", config["width"], config["dropout"])
    return DenseRegressor(n_features, config)


ARTIFACT_DIR = Path("/tmp/task4_regularization")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def fit_candidate(config, tr, va, seed=42, optimizer_kind="builtin", max_epochs=80, patience=12):
    seed_everything(seed)
    pp = FeaturePreprocessor("standard").fit(X[tr])
    xt, xv = pp.transform(X[tr]), pp.transform(X[va])
    ts = TargetScaler(y[tr])
    loader = DataLoader(TensorDataset(xt, ts.encode(y[tr])), batch_size=config["batch_size"],
                        shuffle=True, generator=torch.Generator().manual_seed(seed), num_workers=0)
    model = make_model(config)
    optimizer = build_optimizer(model, optimizer_kind, config["lr"], config["weight_decay"])
    best, best_epoch, stale = math.inf, 0, 0
    history = []
    start = time.perf_counter()
    for epoch in range(1, max_epochs + 1):
        for group in optimizer.param_groups: group["lr"] = learning_rate(config, epoch)
        loss, grad_norm = train_epoch(model, loader, optimizer)
        vp = predict(model, xv, ts)
        mse = regression_metrics(vp, y[va])["mse"]
        history.append(dict(epoch=epoch, train_loss=loss, valid_mse=mse, grad_norm=grad_norm))
        if mse < best:
            best, best_epoch, stale = mse, epoch, 0
            state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if stale >= patience: break
    model.load_state_dict(state)
    vp = predict(model, xv, ts)
    run = dict(id=config["id"], config=dict(config), seed=seed, optimizer=optimizer_kind,
               mse=best, rmse=math.sqrt(best), best_epoch=best_epoch, epochs=len(history),
               seconds=time.perf_counter()-start, history=history)
    prefix = ARTIFACT_DIR / f"{config['id']}_seed{seed}_{optimizer_kind}"
    torch.save(dict(state_dict=state, config=dict(config), target_mean=ts.mean, target_std=ts.std,
                    train_indices=torch.tensor(tr), valid_indices=torch.tensor(va),
                    valid_predictions=vp, best_epoch=best_epoch), str(prefix)+".pt")
    with open(str(prefix)+"_preprocessor.pkl", "wb") as f: pickle.dump(pp, f)
    run["checkpoint"] = str(prefix)+".pt"
    print(f"{config['id']:18s} seed={seed:2d} {optimizer_kind:7s}: MSE={best:.5f}, "
          f"RMSE={math.sqrt(best):.5f}, epoch={best_epoch:2d}/{len(history)}, {run['seconds']:.1f}s", flush=True)
    return run


In [12]:
ANCHOR = {'id': 'bn128_d55_wd0.5', 'architecture': 'dense', 'width': 128, 'depth': 2, 'norm': 'batch', 'activation': 'relu', 'dropout': 0.55, 'input_dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.5, 'batch_size': 256}
ARTIFACT_DIR = Path("/tmp/task4_augmentation")
ARTIFACT_DIR.mkdir(exist_ok=True)

def build_aug_optimizer(model, config, kind="builtin"):
    groups=[{"params":[p for p in model.parameters() if p.ndim>=2],"weight_decay":config["weight_decay"]},
            {"params":[p for p in model.parameters() if p.ndim<2],"weight_decay":0.0}]
    cls = CustomAdamW if kind=="custom" else torch.optim.AdamW
    kwargs={} if kind=="custom" else dict(foreach=False,fused=False)
    return cls(groups,lr=config["lr"],betas=tuple(config["betas"]),**kwargs)


def train_aug_epoch(model,loader,optimizer,config):
    model.train()
    loss_sum,grad_sum,count=0.0,0.0,0
    for features,targets in loader:
        if config["noise"]:
            features=features+config["noise"]*torch.randn_like(features)
        if config["mask"]:
            keep=(torch.rand_like(features)>config["mask"]).to(features.dtype)
            features=features*keep/(1-config["mask"])
        if config["mixup"]:
            alpha=config["mixup"]
            lam=torch.distributions.Beta(alpha,alpha).sample()
            perm=torch.randperm(len(features))
            features=lam*features+(1-lam)*features[perm]
            targets=lam*targets+(1-lam)*targets[perm]
        optimizer.zero_grad(set_to_none=True)
        prediction=model(features)
        loss=nn.functional.mse_loss(prediction,targets)
        assert torch.isfinite(loss)
        loss.backward()
        grad=nn.utils.clip_grad_norm_(model.parameters(),5.0,error_if_nonfinite=True)
        optimizer.step()
        loss_sum+=loss.item()*len(features);grad_sum+=float(grad)*len(features);count+=len(features)
    return loss_sum/count,grad_sum/count


def fit_aug(config,tr,va,seed=42,kind="builtin"):
    seed_everything(seed)
    pp=FeaturePreprocessor("standard").fit(X[tr])
    xt,xv=pp.transform(X[tr]),pp.transform(X[va]);ts=TargetScaler(y[tr])
    loader=DataLoader(TensorDataset(xt,ts.encode(y[tr])),batch_size=config["batch_size"],
                      shuffle=True,generator=torch.Generator().manual_seed(seed),num_workers=0)
    model=make_model(config);optimizer=build_aug_optimizer(model,config,kind)
    best,best_epoch,stale=math.inf,0,0;history=[];started=time.perf_counter()
    for epoch in range(1,161):
        for group in optimizer.param_groups:group["lr"]=learning_rate(config,epoch)
        loss,grad=train_aug_epoch(model,loader,optimizer,config)
        prediction=predict(model,xv,ts);mse=regression_metrics(prediction,y[va])["mse"]
        history.append(dict(epoch=epoch,train_loss=loss,grad_norm=grad,valid_mse=mse))
        if mse<best:
            best,best_epoch,stale=mse,epoch,0;state=copy.deepcopy(model.state_dict())
        else:stale+=1
        if stale>=20:break
    model.load_state_dict(state);prediction=predict(model,xv,ts)
    path=ARTIFACT_DIR/f"{config['id']}_seed{seed}_{kind}.pt"
    torch.save(dict(state_dict=state,config=dict(config),target_mean=ts.mean,target_std=ts.std,
                    valid_indices=torch.tensor(va),train_indices=torch.tensor(tr),
                    valid_predictions=prediction,best_epoch=best_epoch),path)
    with open(str(path).replace(".pt","_preprocessor.pkl"),"wb") as f:pickle.dump(pp,f)
    result=dict(id=config["id"],config=dict(config),seed=seed,optimizer=kind,mse=best,
                rmse=math.sqrt(best),best_epoch=best_epoch,epochs=len(history),history=history,
                checkpoint=str(path),seconds=time.perf_counter()-started)
    print(f"{config['id']:20s} seed{seed}: MSE={best:.6f}, epoch={best_epoch}/{len(history)}",flush=True)
    return result

base_aug=dict(ANCHOR, id="control",mixup=0.0,noise=0.0,mask=0.0,betas=(0.9,0.999))
aug_configs=[base_aug]
for dropout in (0.4,0.55):
    for mixup in (0.05,0.1,0.2,0.4):
        aug_configs.append(dict(base_aug,id=f"mix{mixup}_d{dropout}",dropout=dropout,mixup=mixup))
    for noise in (0.01,0.03,0.1):
        aug_configs.append(dict(base_aug,id=f"noise{noise}_d{dropout}",dropout=dropout,noise=noise))
    for mask in (0.02,0.05,0.1):
        aug_configs.append(dict(base_aug,id=f"mask{mask}_d{dropout}",dropout=dropout,mask=mask))
for betas in ((0.9,0.95),(0.9,0.99),(0.95,0.99),(0.8,0.999)):
    aug_configs.append(dict(base_aug,id=f"beta{betas[0]}_{betas[1]}",betas=betas))
for lr in (0.0003,0.0006,0.002,0.003):
    aug_configs.append(dict(base_aug,id=f"lr{lr}",lr=lr))
for batch in (128,512,1024):
    aug_configs.append(dict(base_aug,id=f"batch{batch}",batch_size=batch))
assert len(aug_configs)==32


## Первичный поиск:31 новая гипотеза и контроль

Mixup использует один коэффициент λ~Beta(α,α) на батч и одновременно смешивает признаки и
нормированные годы внутри train. Gaussian noise добавляется в масштабе стандартизованных признаков.
Маскирование зануляет независимые элементы и масштабирует оставшиеся на 1/(1-p).
Чтобы не ограничить проверку только чрезмерной регуляризацией, каждую аугментацию проверяем
с dropout0.4 и 0.55. Все операции реализованы непосредственно через torch.


In [14]:
aug_runs=[]
for config in aug_configs:
    aug_runs.append(fit_aug(config,train_idx,valid_idx))
    Path("/tmp/task4_augmentation_results.json").write_text(json.dumps(dict(search=aug_runs),indent=2))
aug_ranked=sorted(aug_runs,key=lambda r:r["mse"])
print(pd.DataFrame([r["config"]|{k:r[k] for k in ("seed","mse","rmse","best_epoch","epochs")}
                    for r in aug_ranked]).to_string(index=False))


control              seed42: MSE=80.916541, epoch=35/55
mix0.05_d0.4         seed42: MSE=80.215154, epoch=18/38
mix0.1_d0.4          seed42: MSE=81.560603, epoch=18/38
mix0.2_d0.4          seed42: MSE=80.535680, epoch=33/53
mix0.4_d0.4          seed42: MSE=81.055032, epoch=33/53
noise0.01_d0.4       seed42: MSE=81.044827, epoch=21/41
noise0.03_d0.4       seed42: MSE=80.835091, epoch=29/49
noise0.1_d0.4        seed42: MSE=80.945202, epoch=20/40
mask0.02_d0.4        seed42: MSE=80.869008, epoch=21/41
mask0.05_d0.4        seed42: MSE=80.530672, epoch=21/41
mask0.1_d0.4         seed42: MSE=81.218530, epoch=21/41
mix0.05_d0.55        seed42: MSE=80.331682, epoch=33/53
mix0.1_d0.55         seed42: MSE=81.251443, epoch=30/50
mix0.2_d0.55         seed42: MSE=80.801136, epoch=35/55
mix0.4_d0.55         seed42: MSE=81.125127, epoch=40/60
noise0.01_d0.55      seed42: MSE=81.025121, epoch=30/50
noise0.03_d0.55      seed42: MSE=81.106009, epoch=39/59
noise0.1_d0.55       seed42: MSE=81.350769, epoc

## Подтверждение на seed17 и 83

Три лидера первоначального разбиения зафиксированы до дополнительных запусков. На каждом дополнительном
разбиении заново обучаем также контроль без аугментации и исходный residual-baseline.
Средняя MSE по трём разбиениям используется для ранжирования; audit не оценивается.


In [16]:
finalists=[r for r in aug_ranked if r["id"] != "control"][:3]
print("Зафиксированные финалисты:",[r["id"] for r in finalists])
confirmation_runs=list(finalists)
control_runs=[next(r for r in aug_runs if r["id"]=="control")]
baseline_controls=[]
original_train_idx,original_valid_idx=train_idx.copy(),valid_idx.copy()
baseline_config=dict(name="Residual + standard",architecture="residual",preprocessing="standard",
                     width=128,dropout=0.15,lr=0.001,weight_decay=0.01)
for split_seed in (42,17,83):
    tr,va=train_test_split(development_idx,test_size=2100/11900,random_state=split_seed,
                          stratify=periods[development_idx])
    assert len(tr)==9800 and len(va)==2100 and not (set(tr)|set(va)) & set(audit_idx)
    train_idx,valid_idx=tr,va;y_train,y_valid=y[tr],y[va]
    baseline=fit_experiment(baseline_config,seed=split_seed)
    baseline_controls.append(dict(id="baseline",seed=split_seed,mse=baseline["metrics"]["mse"],
                                  best_epoch=baseline["best_epoch"]))
    if split_seed != 42:
        control_runs.append(fit_aug(base_aug,tr,va,seed=split_seed))
        for leader in finalists:
            confirmation_runs.append(fit_aug(leader["config"],tr,va,seed=split_seed))
train_idx,valid_idx=original_train_idx,original_valid_idx;y_train,y_valid=y[train_idx],y[valid_idx]
summary=[]
for leader in finalists:
    runs=[r for r in confirmation_runs if r["id"]==leader["id"]]
    scores={r["seed"]:r["mse"] for r in runs}
    baseline_scores={r["seed"]:r["mse"] for r in baseline_controls}
    control_scores={r["seed"]:r["mse"] for r in control_runs}
    summary.append(dict(id=leader["id"],mean_mse=sum(scores.values())/3,
                        mse42=scores[42],mse17=scores[17],mse83=scores[83],
                        delta_baseline=sum(scores[s]-baseline_scores[s] for s in scores)/3,
                        delta_control=sum(scores[s]-control_scores[s] for s in scores)/3,
                        median_epoch=int(torch.tensor([r["best_epoch"] for r in runs]).median())))
summary.sort(key=lambda r:r["mean_mse"])
print("Исходный baseline:");print(pd.DataFrame(baseline_controls).to_string(index=False))
print("Контроль без аугментации:")
print(pd.DataFrame([{k:r[k] for k in ("id","seed","mse","best_epoch")} for r in control_runs]).to_string(index=False))
print("Лидеры:");print(pd.DataFrame(summary).to_string(index=False))
Path("/tmp/task4_augmentation_results.json").write_text(json.dumps(dict(search=aug_runs,
    confirmation=confirmation_runs,baseline_controls=baseline_controls,control_runs=control_runs,
    summary=summary),indent=2))


Зафиксированные финалисты: ['mix0.05_d0.4', 'mix0.05_d0.55', 'mask0.05_d0.4']
Residual + standard / builtin: epoch=  1, train RMSE=9.0943, valid RMSE=9.4623
Residual + standard / builtin: epoch= 20, train RMSE=4.7133, valid RMSE=10.0868
Лучший checkpoint (4 эпоха): RMSE=9.1591; MSE=83.8891; MAE=6.2957
Residual + standard / builtin: epoch=  1, train RMSE=9.2507, valid RMSE=9.3099
Residual + standard / builtin: epoch= 20, train RMSE=4.8774, valid RMSE=9.9723
Лучший checkpoint (4 эпоха): RMSE=8.9243; MSE=79.6431; MAE=6.1630
control              seed17: MSE=76.321704, epoch=31/51
mix0.05_d0.4         seed17: MSE=77.443489, epoch=17/37
mix0.05_d0.55        seed17: MSE=76.884662, epoch=27/47
mask0.05_d0.4        seed17: MSE=77.056574, epoch=29/49
Residual + standard / builtin: epoch=  1, train RMSE=9.0409, valid RMSE=9.4801
Residual + standard / builtin: epoch= 20, train RMSE=4.7255, valid RMSE=10.4380
Лучший checkpoint (3 эпоха): RMSE=9.3353; MSE=87.1480; MAE=6.5107
control              see

## Custom AdamW и финальные одиночные модели

Повторяем лучшего кандидата на исходном разбиении с собственным AdamW и сравниваем прогнозы.
Каждый из трёх финалистов отдельно обучается на 14000 строках по медиане лучших эпох трёх разбиений.
Все training-аугментации сохраняются при refit; prediction проходит в eval mode без аугментаций.


In [18]:
winner=next(r for r in finalists if r["id"]==summary[0]["id"])
custom=fit_aug(winner["config"],train_idx,valid_idx,kind="custom")
a=torch.load(winner["checkpoint"],weights_only=True)["valid_predictions"]
b=torch.load(custom["checkpoint"],weights_only=True)["valid_predictions"]
max_difference=(a-b).abs().max().item()
print("Custom - builtin MSE:",custom["mse"]-winner["mse"],"max prediction delta:",max_difference)
assert abs(custom["mse"]-winner["mse"])<1e-4
assert custom["best_epoch"]==winner["best_epoch"]
submission_dir=DATA_DIR/"Submission";submission_dir.mkdir(exist_ok=True)
final_outputs=[]
for result in summary:
    config=next(r["config"] for r in finalists if r["id"]==result["id"])
    seed_everything(42);pp=FeaturePreprocessor("standard").fit(X);ts=TargetScaler(y)
    features=pp.transform(X)
    loader=DataLoader(TensorDataset(features,ts.encode(y)),batch_size=config["batch_size"],
                      shuffle=True,generator=torch.Generator().manual_seed(42),num_workers=0)
    model=make_model(config);optimizer=build_aug_optimizer(model,config,"custom")
    for epoch in range(1,result["median_epoch"]+1):
        for group in optimizer.param_groups:group["lr"]=learning_rate(config,epoch)
        train_aug_epoch(model,loader,optimizer,config)
    prediction=predict(model,pp.transform(X_test),ts)
    submission=pd.DataFrame({"id":test_ids.to_numpy(),"year":prediction.numpy()})
    assert len(submission)==6000 and list(submission.columns)==["id","year"]
    assert submission.id.equals(test_ids.reset_index(drop=True)) and torch.isfinite(prediction).all()
    output=submission_dir/f"task4_augmentation_{result['id']}.csv"
    submission.to_csv(output,index=False,float_format="%.6f")
    check=pd.read_csv(output)
    assert check.id.equals(test_ids.reset_index(drop=True))
    torch.testing.assert_close(torch.tensor(check.year.to_numpy()),prediction.double(),rtol=0,atol=1e-6)
    torch.save(dict(state_dict=model.state_dict(),config=config,target_mean=ts.mean,target_std=ts.std,
                    epochs=result["median_epoch"],test_predictions=prediction),ARTIFACT_DIR/f"{result['id']}_final.pt")
    with open(ARTIFACT_DIR/f"{result['id']}_final_preprocessor.pkl","wb") as f:pickle.dump(pp,f)
    final_outputs.append(dict(id=result["id"],path=str(output),rows=len(submission),epochs=result["median_epoch"],
                              sha256=hashlib.sha256(output.read_bytes()).hexdigest()))
    print(output,"epochs",result["median_epoch"],"range",float(prediction.min()),float(prediction.max()))
Path("/tmp/task4_augmentation_results.json").write_text(json.dumps(dict(search=aug_runs,
    confirmation=confirmation_runs,baseline_controls=baseline_controls,control_runs=control_runs,
    summary=summary,custom_comparison=custom,custom_max_prediction_difference=max_difference,
    submissions=final_outputs),indent=2))


mix0.05_d0.55        seed42: MSE=80.331682, epoch=33/53
Custom - builtin MSE: 0.0 max prediction delta: 0.0
homeworks/task4_data/Submission/task4_augmentation_mix0.05_d0.55.csv epochs 27 range 1977.0816650390625 2006.509033203125
homeworks/task4_data/Submission/task4_augmentation_mask0.05_d0.4.csv epochs 21 range 1975.1466064453125 2008.97412109375
homeworks/task4_data/Submission/task4_augmentation_mix0.05_d0.4.csv epochs 18 range 1973.0216064453125 2007.659912109375


## Вывод

Проверена **31 новая гипотеза** и повторён контроль. Лучшие три конфигурации по первоначальному
validation подтверждены на seed17/83. Средняя MSE без аугментации составила **80.4038**, а лучшие
альтернативы дали **80.4695** (mixup0.05, dropout0.55), **80.5037** (mask0.05, dropout0.4) и
**80.5860** (mixup0.05, dropout0.4). Таким образом, заметное улучшение первоначального validation
**не воспроизвелось в среднем на дополнительных разбиениях**. Эти результаты не доказывают пользу
аугментации по сравнению с сильной обычной MLP; в качестве основного варианта сохраняем контроль.

Три CSV оставлены как отдельные альтернативы для добровольной проверки public score.
Они получены от отдельных одиночных сетей, никакого объединения прогнозов нет.
Собственный AdamW совпал со встроенным. Все три split уже участвуют в подборе; они не являются
независимым финальным тестом. Audit не оценивался, скрытые Kaggle-метки недоступны.


## Приложение: однократная контрольная оценка после фиксации кандидатов

Все архитектуры, гиперпараметры, кандидаты и submission уже зафиксированы до этой ячейки.
После завершения общего поиска разрешена одна оценка исходных checkpoint seed42 на 2100 строках audit.
Модели для этой оценки обучались только на 9800 training-строках, preprocessing восстанавливается по тем же
строкам. Финальные модели, обученные на 14000, здесь не оцениваются: audit входит в их обучение.

Этот audit **уже использовался в самом первом baseline-решении**. В текущем поиске его метрики и
предсказания до этой ячейки не вычислялись; поэтому это проверка после подбора, но её нельзя называть
совершенно новым, ранее никому не известным тестом. После проверки настройки не меняются и поиск не продолжается.
Значения отражают локальные данные и не являются public/private Kaggle score.


In [21]:
audit_records = []
audit_prediction_payload = {"audit_indices":torch.tensor(audit_idx),"predictions":{},
                            "targets":y[audit_idx]}
frozen_ids = [row["id"] for row in summary]
for candidate_id in frozen_ids:
    run = next(r for r in confirmation_runs if r["id"]==candidate_id and r["seed"]==42)
    checkpoint = torch.load(run["checkpoint"],weights_only=True,map_location="cpu")
    source_train_idx = checkpoint["train_indices"].numpy()
    assert len(source_train_idx)==9800
    assert not set(source_train_idx)&set(audit_idx)
    pp = FeaturePreprocessor("standard").fit(X[source_train_idx])
    ts = TargetScaler(y[source_train_idx])
    assert abs(ts.mean-checkpoint["target_mean"])<1e-9
    assert abs(ts.std-checkpoint["target_std"])<1e-9
    model = make_model(checkpoint["config"])
    model.load_state_dict(checkpoint["state_dict"])
    audit_prediction = predict(model,pp.transform(X[audit_idx]),ts)
    metrics = regression_metrics(audit_prediction,y[audit_idx])
    audit_records.append(dict(id=candidate_id,seed=42,best_epoch=checkpoint["best_epoch"],
                              checkpoint=run["checkpoint"],**metrics))
    audit_prediction_payload["predictions"][candidate_id] = audit_prediction
print("Audit after selection; previously used by the original baseline:")
print(pd.DataFrame(audit_records).drop(columns="checkpoint").to_string(index=False))
audit_report = dict(frozen_candidates=frozen_ids,after_selection=True,
                    audit_previously_used_by_original_baseline=True,n_audit=len(audit_idx),
                    audit_used_during_current_search=False,records=audit_records)
Path("/tmp/task4_augmentation_audit.json").write_text(json.dumps(audit_report,indent=2))
torch.save(audit_prediction_payload,Path("/tmp/task4_augmentation/audit_predictions.pt"))


Audit after selection; previously used by the original baseline:
           id  seed  best_epoch     rmse       mse      mae
mix0.05_d0.55    42          33 9.136503 83.475688 6.362934
mask0.05_d0.4    42          21 9.123951 83.246479 6.387040
 mix0.05_d0.4    42          18 9.080180 82.449669 6.301183
